# Install Pixi Environments in OpenSciencelab using pixi.toml

- Pixi environments are isolated Python software environments. 

- They allow you to install and use multiple software configurations that would conflict with each other in a single environment.   

In [ ]:
import os
from contextlib import contextmanager
from pathlib import Path
import ipywidgets
import fileinput

try:
    from ipyfilechooser import FileChooser
except:
    !python -m pip install ipyfilechooser
    from ipyfilechooser import FileChooser

@contextmanager
def work_dir(new_dir):
    old_dir = os.getcwd()
    os.chdir(new_dir)
    try:
        yield
    finally:
        os.chdir(old_dir)

<hr>

### Select a Pixi Environment to Create

In [ ]:
environments = list((Path.cwd() / "Pixi_Environments").glob("*/*.toml"))

In [ ]:
chosen_env = ipywidgets.RadioButtons(
    options=[i.parent.name for i in environments] + ["Custom Environment"],
    description="",
    disabled=False,
    layout=ipywidgets.Layout(width='1000px')
)
display(chosen_env)

<hr>

### Build Pixi Environment

In [ ]:
# Prompt for custom environment if selected
if "custom" in chosen_env.value.lower():
    print("Select your environment's directory")
    fc = FileChooser(Path.cwd(), directory=True)
    display(fc)

In [ ]:
# Resolve env_path
if "custom" in chosen_env.value.lower():
    env_path = fc.selected
    fileinput.close()
    env_name = Path(fc.selected).name
else:
    env_path = Path.cwd() / "Pixi_Environments" / chosen_env.value
    env_name = chosen_env.value

In [ ]:
with work_dir(env_path):
    !pixi install

<hr>

### Add Environment to ipykernel

In [ ]:
display_name = f'"{env_name} (Python)"'

with work_dir(env_path):
    !pixi run -e default python -m ipykernel install \
      --user \
      --name $env_name \
      --display-name $display_name
